# Workspace for Fourier Tasks

- By: (redacted), 2025
- Corresponding paper (redacted), 2026
- Licence: MIT
- [Click here for the Fourier source repository (redacted)](https://fake.url)

---

## Background

We can analyse rhythms using DFT.

We provide a corpus of eight-measure ragtime melodic rhythms in taken from Yust and Kirlin 2021.

This corpus consists of 71 pieces composed by the 'big three' ragtime composers:
Scott Joplin, James Scott, and Joseph Lamb.

Each piece has four or more strains for a total of 268 rhythms.
The rhythms are given as indicator vectors,
and are based just on the onsets of the right-hand melody,
ignoring the left hand which usually plays an oom-pah pattern in straight crotchets.

The corpus includes only the first eight measures of each strain, discarding measures 9 to 16.
Each measure is 8 sixteenth notes in length (2/4), so the total length of each vector is 64.

## Explore

Use the cells below to explore this data and the fact stated above before we begin the tasks.

In [ ]:
import pandas as pd

Chose a path (local or remote) and import the data:

In [ ]:
# If you have the whole repo stored locally then use this (and comment out the alternative below)
path_to_data = './data/'

# If not, use the URL:
path_to_data = "https://raw.githubusercontent.com/music-computing/fourier/refs/heads/main/data/"

# Either way, the import is the same:
df = pd.read_csv(path_to_data + "ragtime8m_data.csv", header=0)

In [ ]:
rhythm_names = df["Piece"]  # Store for later ...

In [ ]:
# ... For now just check that there are 268 entries
len(rhythm_names)

In [ ]:
# Filters for numeric (i.e., the vectors) and convert to array
df_numeric = df.select_dtypes(include='number')
rhythm_vectors = df_numeric.values

In [ ]:
# The vector part of each entry is of len 64
for row in rhythm_vectors:
    assert len(row) == 64

## Task

1. Convert all rhythms via DFT
2. Convert the DFTs to polar coordinates
3. Find the average and standard deviations of $|\hat{x}_1|$-$|\hat{x}_{32}|$
4. Plot the averages and averages $+/-$ the standard deviations
5. Now go back to the original complex numbers and average the real and imaginary values across the corpus for $\hat{x}_1$--$\hat{x}_{32}$
6. Convert these averaged coefficient values to complex numbers
7. Plot the magnitudes of these along with averages from steps (3)--(4)

Bonus:
The paper proposes several additional questions which are also explored here.

---

## Workspace

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft

# Set up plotting style (optional)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

---

## Solutions

### Solution 1: Convert all rhythms via DFT

In [ ]:
dft_coefficients = np.array([fft(rhythm) for rhythm in rhythm_vectors])

print(f"Computed DFT for {len(dft_coefficients)} rhythms")
print(f"DFT coefficients shape: {dft_coefficients.shape}")

### Solution 2: Convert DFTs to polar coordinates

In [ ]:
# Convert all DFTs to polar coordinates
all_magnitudes, all_phases = np.abs(dft_coefficients), np.angle(dft_coefficients)

print(f"Magnitudes shape: {all_magnitudes.shape}")
print(f"Phases shape: {all_phases.shape}")
print(f"\nExample: first 4 magnitudes of the first rhythm:\n{all_magnitudes[0, :4]}")

### Solution 3: Find average and standard deviations of $|\hat{x}_1|$-$|\hat{x}_{32}|$

Focus on coefficients 1 to 32.

As we're 0-indexed: 1 to 32 corresponds to indices [1:33].

For a refresher on this, try the [`invariance` notebook](https://github.com/music-computing/fourier/blob/main/invariance.ipynb).

In [ ]:
coeff_indices = np.arange(1, 33)  # softcoded in case you want to experiment with others

# Extract magnitudes for indices 1-32
magnitudes_subset = all_magnitudes[:, coeff_indices]

# Compute mean and standard deviation across the corpus
mean_magnitudes = np.mean(magnitudes_subset, axis=0)
std_magnitudes = np.std(magnitudes_subset, axis=0)

print("Statistics for coefficients 1–32:")
print(f"Mean magnitudes shape: {mean_magnitudes.shape}")
print(f"\nMean magnitudes:")
print(mean_magnitudes)
print(f"\nStandard deviations:")
print(std_magnitudes)

### Solution 4: Plot averages and averages ± standard deviations

In [ ]:
def plot_averages_with_std(coeff_indices, mean_mags, std_mags, title="Magnitude Spectrum with Error Bars"):
    """
    Plot the average magnitudes with error bars showing ±1 standard deviation.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot mean
    ax.plot(coeff_indices, mean_mags, 'b-', linewidth=2, label=r'Mean $|\hat{x}_k|$')
    
    # Plot error bands (±1 std)
    ax.fill_between(coeff_indices, 
                    mean_mags - std_mags, 
                    mean_mags + std_mags, 
                    alpha=0.3, color='blue', label='+/- 1 Standard Deviation')
    
    ax.set_xlabel('Coefficient Index (k)', fontsize=12)
    ax.set_ylabel(r'Magnitude $|\hat{x}_k|$', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_xticks(coeff_indices)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    # plt.savefig("./plots/ragtime_dft.pdf")  # Uncomment this line if you'd like to save a copy
    plt.show()

# Plot the results
plot_averages_with_std(
    coeff_indices, mean_magnitudes, std_magnitudes,
    title="Average DFT Magnitudes with Standard Deviation (Coefficients 1-32)"
)

### Solution 5: Average real and imaginary values across the corpus

In [ ]:
# Extract real and imaginary parts of DFT coefficients
dft_subset = dft_coefficients[:, coeff_indices]

# Average real parts and imaginary parts separately
mean_real = np.mean(dft_subset.real, axis=0)
mean_imag = np.mean(dft_subset.imag, axis=0)

print("Averaged complex number components (coefficients 1-32):")
print(f"Mean Real parts: {mean_real}")
print(f"Mean Imaginary parts: {mean_imag}")

### Solution 6: Convert averaged coefficients to complex numbers

In [ ]:
# Convert averaged real and imaginary parts back to complex numbers
averaged_complex = mean_real + 1j * mean_imag

# Compute magnitudes of these averaged complex numbers
averaged_magnitudes = np.abs(averaged_complex)

print("Averaged complex coefficients:")
print(f"Complex values: {averaged_complex}")
print(f"\nMagnitudes of averaged coefficients:")
print(averaged_magnitudes)

### Solution 7: Plot magnitudes alongside averages from steps (3)-(4)

In [ ]:
def plot_comparison(
        coeff_indices, polar_avg_mags, complex_avg_mags,
        title="Comparison: Polar Averaging vs Complex Averaging"
):
    """
    Plot both averaging methods on the same figure for comparison.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot polar averaging (mean of magnitudes)
    ax.plot(
        coeff_indices, polar_avg_mags, 'b-o', linewidth=2, markersize=6,
        label=r'Polar: Mean($|\hat{x}_k|$)'
    )
    
    # Plot complex averaging (magnitude of mean complex)
    ax.plot(
        coeff_indices, complex_avg_mags, 'r-s', linewidth=2, markersize=6,
        label=r'Complex: $|Mean(\hat{x}_k)|$'
    )
    
    ax.set_xlabel('Coefficient Index (k)', fontsize=12)
    ax.set_ylabel('Magnitude', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_xticks(coeff_indices)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    # plt.savefig("./plots/ragtime_comparison.pdf")  # Uncomment this line if you'd like to save a copy
    plt.show()

# Plot the comparison
plot_comparison(coeff_indices, mean_magnitudes, averaged_magnitudes)

---

## Discussion Questions

### Q1: Which coefficient is most dominant?

In [ ]:
max_idx_polar   = np.argmax(mean_magnitudes)
max_idx_complex = np.argmax(averaged_magnitudes)

print(f"Polar averaging:   k={coeff_indices[max_idx_polar]},  mean magnitude={mean_magnitudes[max_idx_polar]:.4f}")
print(f"Complex averaging: k={coeff_indices[max_idx_complex]}, mean magnitude={averaged_magnitudes[max_idx_complex]:.4f}")

# Prototype: rhythm closest to the complex average
distances = np.linalg.norm(dft_coefficients[:, coeff_indices] - averaged_complex, axis=1)
proto_idx = np.argmin(distances)
print(f"Prototype (closest to average): {rhythm_names[proto_idx]}")

Both methods agree: **k=24** is dominant. k=24 corresponds to three cycles per measure (each measure = 8 sixteenth notes, so period = 8/3 sixteenth notes) — the characteristic syncopated subdivision of ragtime.


### Q2: Does the spectrum reflect repetition or oversampling?

In [ ]:
# Peaks at integer multiples of 8 would indicate exact repetition every measure
harmonics = [8, 16, 24, 32]
for k in harmonics:
    print(f"k={k:2d}: mean magnitude = {mean_magnitudes[k-1]:.4f}")

The spectrum peaks at k=24.
This rules out smooth oversampling (which would concentrate energy at very low k).
It also rules out simple regular-division repertoire which would give equal-height peaks at k=8, 16, 24, 32.
Instead, the dominant periodicity is at the level of the syncopated subdivision.
This means, we're looking at a pattern, but one of a length not equal to the measure, half-measure etc.

### Q3: How do the two averaging methods compare?

In [ ]:
def circular_std(angles):
    """Lower = more phase-coherent across the corpus."""
    r = np.sqrt(np.mean(np.cos(angles))**2 + np.mean(np.sin(angles))**2)
    return np.sqrt(-2 * np.log(r))

print(f"{'k':>4}  {'polar avg':>10}  {'complex avg':>12}  {'difference':>10}  {'phase circ_std':>15}")
for k in [8, 16, 24, 32]:
    i = k - 1
    diff = abs(mean_magnitudes[i] - averaged_magnitudes[i])
    cs   = circular_std(all_phases[:, k])
    print(f"{k:>4}  {mean_magnitudes[i]:>10.4f}  {averaged_magnitudes[i]:>12.4f}  {diff:>10.4f}  {cs:>15.4f}")

Polar averaging ignores phase and sums magnitudes directly.
Complex averaging allows phase cancellation before taking the magnitude, so it always yields a smaller or equal value.

The two methods differ at every coefficient because phase is never fully coherent across 268 examples.

The smallest circular standard deviation (most coherent phases) occurs at $k=24$.
This was our dominant coefficient (see Q1).
That means that the corpus agrees not just on *how much* syncopation, but on *where* in the cycle it falls.


End

---